In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/tips-hindawi-university-info/Tips Hindawi University Info.pdf


In [2]:
!pip install sentence-transformers PyPDF2 faiss-cpu
from sentence_transformers import SentenceTransformer
from PyPDF2 import PdfReader
import faiss
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.6 MB/s eta 0:00:00:00:0100:01


2026-02-08 01:14:16.103588: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770513256.326818      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770513256.391954      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770513256.935379      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770513256.935431      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770513256.935433      55 computation_placer.cc:177] computation placer alr

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load model and tokenizer
model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")


def generate_text(prompt, max_length=100, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

model-00004-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [4]:
def extract_text(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

In [5]:
def chunk_text(text, size = 500, overlap = 50):
    words = text.split()
    chunks = []
    for i in range (0, len(words), (size - overlap)):
        chunk = " ".join(words[i:i + size])
        chunks.append(chunk)
    return chunks

In [6]:
def embedded_chunks(chunks, model_name = 'sentence-transformers/all-MiniLM-L6-v2'):
    model = SentenceTransformer(model_name)
    embedding = model.encode(chunks, convert_to_numpy = True)
    return model, embedding

In [7]:
def faiss_index(embedding):
    dim = embedding.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embedding)
    return index

In [9]:
def search(query, model, index, chunks, k=1):
    query_embedding = model.encode([query], convert_to_numpy = True)
    distances, indices = index.search(query_embedding , k)
    return [chunks[i] for i in indices[0]]

In [10]:
pdf_path = '/kaggle/input/tips-hindawi-university-info/Tips Hindawi University Info.pdf'
text = extract_text(pdf_path)
chunks = chunk_text(text, size = 50, overlap = 10)
model_embedding, embedding = embedded_chunks(chunks)
index = faiss_index(embedding)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
question = "Where is Tips Hindawi University located?"
top_chunks = search(question, model_embedding, index, chunks, k=1)

for i, chunk in enumerate(top_chunks, 1):
    print(f"\n--- Chunk {i} ---\n{chunk}")


--- Chunk 1 ---
1. General Overview Tips Hindawi University (THU) is a premier institution of higher education located in the heart of the Middle East. Founded in 1963, the university has grown into a globally recognized center for academic excellence and innovation. With over six decades of educational leadership, THU has produced more


In [13]:
question = "Where is Tips Hindawi University located?"
chunk = top_chunks[0]

prompt = f"Answer the next question: {question} by reading the following text:{chunk}"

In [14]:
answer = generate_text(prompt, max_length=200)
print(answer[0])

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:2532: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


Answer the next question: Where is Tips Hindawi University located? by reading the following text:1. General Overview Tips Hindawi University (THU) is a premier institution of higher education located in the heart of the Middle East. Founded in 1963, the university has grown into a globally recognized center for academic excellence and innovation. With over six decades of educational leadership, THU has produced more than 100,000 alumni who have made significant contributions to various fields, both locally and internationally.

2. Campus and Facilities
THU's main campus is situated on a sprawling 200-acre site in the bustling city of Cairo, Egypt. The campus is designed to be a self-sustaining academic community, with state-of-the-art facilities, lush greenery, and modern infrastructure. It houses 15 faculties and 40 research centers, providing a comprehensive range of academic programs and research opportunities. The campus also


In [15]:
question = "Does the university offer online programs?"
top_chunks = search(question, model_embedding, index, chunks, k=1)

for i, chunk in enumerate(top_chunks, 1):
    print(f"\n--- Chunk {i} ---\n{chunk}")


--- Chunk 1 ---
globally. The THU Alumni Network hosts quarterly webinars, mentorship programs, and reunions. End of document• • • 4


In [18]:
question = "Does the university offer online programs?"
chunk = top_chunks[0]

prompt = f"Answer the next question: {question} by reading the following text:{chunk}"

In [19]:
answer = generate_text(prompt, max_length=200)
print(answer[0])

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Answer the next question: Does the university offer online programs? by reading the following text:globally. The THU Alumni Network hosts quarterly webinars, mentorship programs, and reunions. End of document• • • 4. The university offers online programs.


In [21]:
question = "Is there financial aid for international students?"
top_chunks = search(question, model_embedding, index, chunks, k=1)

for i, chunk in enumerate(top_chunks, 1):
    print(f"\n--- Chunk {i} ---\n{chunk}")


--- Chunk 1 ---
Admissions Relevant bachelor’s degree Minimum GPA of 3.0/4.0 Two academic references Statement of purpose 4.3 Tuition Fees (Annual) Undergraduate: $5,000 - $8,000 Graduate: $6,500 - $10,000 4.4 Scholarships Merit Scholarships : 25% to 100% tuition coverage Need-Based Grants Research Fellowships for graduate students 5. Administration President : Dr . Nabil


In [23]:
question = "Is there financial aid for international students?"
chunk = top_chunks[0]

prompt = f"Answer the next question: {question} by reading the following text:{chunk}"

In [24]:
answer = generate_text(prompt, max_length=200)
print(answer[0])

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Answer the next question: Is there financial aid for international students? by reading the following text:Admissions Relevant bachelor’s degree Minimum GPA of 3.0/4.0 Two academic references Statement of purpose 4.3 Tuition Fees (Annual) Undergraduate: $5,000 - $8,000 Graduate: $6,500 - $10,000 4.4 Scholarships Merit Scholarships : 25% to 100% tuition coverage Need-Based Grants Research Fellowships for graduate students 5. Administration President : Dr . Nabil Al-Najjar Provost : Dr . Fatima Al-Hajjar Dean of Admissions : Dr . Amal Al-Rahman 6. Contact Information Address : 123 Main Street, Anytown, Anycountry Phone : +1 (123) 456-7890 Email : admissions


In [25]:
question = "What languages are used for instruction?"
top_chunks = search(question, model_embedding, index, chunks, k=1)

for i, chunk in enumerate(top_chunks, 1):
    print(f"\n--- Chunk {i} ---\n{chunk}")


--- Chunk 1 ---
Faculty Highlights Dr. Yasir Al-Sabbagh (Computer Science): Expert in natural language processing Prof. Rana Khalil (Economics): World Bank consultant and author Dr. Omar Al-Mansoori (Mechanical Engineering): Holder of 11 patents• • • • • • • • • • • • • • • • • • • • •


In [26]:
question = "What languages are used for instruction?"
chunk = top_chunks[0]

prompt = f"Answer the next question: {question} by reading the following text:{chunk}"

In [27]:
answer = generate_text(prompt, max_length=200)
print(answer[0])

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Answer the next question: What languages are used for instruction? by reading the following text:Faculty Highlights Dr. Yasir Al-Sabbagh (Computer Science): Expert in natural language processing Prof. Rana Khalil (Economics): World Bank consultant and author Dr. Omar Al-Mansoori (Mechanical Engineering): Holder of 11 patents• • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • • •
